## 1️.Install Libraries

In [1]:
# Run once — restart kernel after installing
# !pip install pdfplumber pytesseract pillow pandas openpyxl google-generativeai reportlab

## 2️.Configuration

In [11]:
import os

os.environ["GEMINI_API_KEY"] = ""  # ← Paste your key here

EXCEL_FILE     = "gst_data.xlsx"   # Output Excel file
INVOICE_FOLDER = "invoices/"       # Folder for batch mode

print("Config loaded ")

Config loaded 


## 3️.Text Extraction

In [12]:
import pdfplumber
from PIL import Image, ImageEnhance, ImageFilter
import pytesseract
import os

SUPPORTED_EXTENSIONS = {".pdf", ".png", ".jpg", ".jpeg", ".tiff", ".tif", ".bmp", ".webp"}


def preprocess_image(image):
    image = image.convert("L")
    image = ImageEnhance.Contrast(image).enhance(2.0)
    image = image.filter(ImageFilter.SHARPEN)
    return image


def table_to_text(table):
    lines = []
    for row in table:
        cells = [str(c).strip() if c else "" for c in row]
        non_empty = [c for c in cells if c]
        if not non_empty:
            continue
        if len(cells) == 2:
            for c in cells:
                if c: lines.append(c)
        elif len(cells) == 4 and any(":" in c for c in cells):
            for i in range(0, len(cells), 2):
                k = cells[i].strip()
                v = cells[i+1].strip() if i+1 < len(cells) else ""
                if k or v: lines.append(f"{k} {v}".strip())
        else:
            lines.append("  |  ".join(non_empty))
    return "\n".join(lines)


def extract_text_from_pdf(file_path):
    parts = []
    with pdfplumber.open(file_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            raw_text   = page.extract_text() or ""
            tables     = page.extract_tables()
            table_texts = [table_to_text(t) for t in tables if t]

            if not raw_text.strip() and not table_texts:  # scanned page
                pil = page.to_image(resolution=300).original
                parts.append(pytesseract.image_to_string(preprocess_image(pil), config="--psm 6 --oem 3"))
                continue

            page_text = raw_text + "\n"
            if table_texts:
                page_text += "\n--- STRUCTURED TABLE DATA ---\n" + "\n\n".join(table_texts)
            parts.append(page_text)
    return "\n\n".join(parts).strip()


def extract_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    if ext not in SUPPORTED_EXTENSIONS:
        raise ValueError(f"Unsupported file type: '{ext}'")
    if ext == ".pdf":
        return extract_text_from_pdf(file_path)
    return pytesseract.image_to_string(preprocess_image(Image.open(file_path)), config="--psm 6 --oem 3").strip()


# ── Quick test ──
text = extract_text("invoice_1_electronics.pdf")
print(f"Extracted {len(text)} characters\n", text[:500])

Extracted 1455 characters
 TAX INVOICE
Seller: Invoice Details
Sri Murugan Electronics Pvt Ltd Invoice No: SME/2025-26/1042
No. 45, Anna Salai, Chennai - 600002 Invoice Date: 05-01-2026
Tamil Nadu Place of Supply: Tamil Nadu (33)
GSTIN: 33AABCS1429B1Z8
Phone: 044-24356789
Bill To:
Mr. Rajesh Kumar
12, Gandhi Nagar, Coimbatore - 641012
GSTIN: 33BBBPK5678M1Z3
S.No Description HSN Qty Unit Rate (Rs.) Amount (Rs.)
1 Samsung 55" 4K Smart TV 8528 1 Nos 42,500.00 42,500.00
2 LG Soundbar HW-Q60T 8518 1 Nos 8,999.00 8,999.00
3 HDM


## 4️.LLM Data Extraction

In [13]:
import google.generativeai as genai
import json, re, time

genai.configure(api_key=os.environ["GEMINI_API_KEY"])
model = genai.GenerativeModel("gemini-2.5-flash")

EXTRACTION_PROMPT = """
You are an expert GST invoice data extraction engine for Indian businesses.

The invoice text may contain:
- A "STRUCTURED TABLE DATA" section with pipe-separated columns — prefer this for line items
- Indian number formats like "4,20,000" (= 420000) or "Rs. 1,07,100" (= 107100)
- Both CGST+SGST (intra-state) OR only IGST (inter-state) — never both

Return STRICTLY valid JSON only. No markdown, no explanation, no backticks.

{{
  "vendor_name": "Full legal name of the SELLER (NOT the buyer/bill-to)",
  "gstin": "SELLER GSTIN — exactly 15 chars like 29ABCDE1234F1Z5, null if not found",
  "invoice_number": "Invoice or bill number",
  "invoice_date": "DD-MM-YYYY",
  "taxable_amount": <number>,
  "cgst": <number — 0 if absent>,
  "sgst": <number — 0 if absent>,
  "igst": <number — 0 if absent>,
  "total_amount": <number — grand total including ALL taxes>
}}

RULES:
1. vendor_name + gstin: the SELLER at the top. NOT the Bill To / buyer.
2. gstin: must be exactly 15 chars. Return null if invalid or missing.
3. Indian numbers: remove ALL commas. "4,20,000" → 420000.
4. total_amount: use GRAND TOTAL / TOTAL INVOICE VALUE / TOTAL AMOUNT line.
5. CGST+SGST for intra-state. IGST for inter-state. Set 0 for any absent tax.
6. All numeric fields must be plain float/int — NOT strings.

Invoice Text:
{text}
"""


def parse_indian_number(value):
    """'4,20,000.00' → 420000.0"""
    if value is None: return 0.0
    s = re.sub(r"[^\d.]", "", str(value).replace(",", ""))
    try: return float(s) if s else 0.0
    except ValueError: return 0.0


def extract_invoice_data(text, max_retries=3):
    """Extract structured data from invoice text. Auto-retries on rate limit (429)."""
    if not text or len(text.strip()) < 10:
        print("  Warning: Text too short."); return None

    for attempt in range(1, max_retries + 1):
        try:
            response = model.generate_content(EXTRACTION_PROMPT.format(text=text))
            raw = response.text or response.candidates[0].content.parts[0].text
            cleaned = raw.strip()

            if "```" in cleaned:
                for part in cleaned.split("```"):
                    s = part.strip().lstrip("json").strip()
                    if s.startswith("{"): cleaned = s; break

            data = json.loads(cleaned.strip())
            for f in ["taxable_amount", "cgst", "sgst", "igst", "total_amount"]:
                data[f] = parse_indian_number(data.get(f))
            return data

        except Exception as e:
            err = str(e)
            if "429" in err or "quota" in err.lower():
                # Extract suggested wait time from error message
                m = re.search(r'seconds.*?(\d+)', err)
                wait = int(m.group(1)) + 5 if m else 15 * attempt
                if attempt < max_retries:
                    print(f"  ⏳ Rate limit — waiting {wait}s (retry {attempt}/{max_retries})...")
                    time.sleep(wait); continue
                else:
                    print(f"  ✘ Rate limit — all retries exhausted. Try again in 1 min."); return None
            elif isinstance(e, json.JSONDecodeError):
                if attempt < max_retries: time.sleep(3); continue
                return None
            else:
                print(f"  LLM error: {e}"); return None
    return None


# ── Quick test ──
data = extract_invoice_data(extract_text("invoice_1_electronics.pdf"))
print(data)

{'vendor_name': 'Sri Murugan Electronics Pvt Ltd', 'gstin': '33AABCS1429B1Z8', 'invoice_number': 'SME/2025-26/1042', 'invoice_date': '05-01-2026', 'taxable_amount': 52399.0, 'cgst': 4715.91, 'sgst': 4715.91, 'igst': 0.0, 'total_amount': 61831.0}


## 5️.Validation

In [7]:
import re, os, pandas as pd
from datetime import datetime

# v3 FIX: correct GSTIN pattern — allows letters at check digit position (e.g. Q, V, P)
GSTIN_PATTERN = r"^\d{2}[A-Z]{5}\d{4}[A-Z]{1}[1-9A-Z]{1}Z[0-9A-Z]{1}$"


def validate_gstin(gstin):
    if not gstin: return None
    g = str(gstin).strip().upper()
    if len(g) != 15: return None
    return g if re.match(GSTIN_PATTERN, g) else None


def validate_date(date_str):
    if not date_str: return None
    for fmt in ("%d-%m-%Y", "%d/%m/%Y", "%Y-%m-%d", "%d %b %Y", "%d %B %Y"):
        try: return datetime.strptime(str(date_str).strip(), fmt).strftime("%d-%m-%Y")
        except ValueError: continue
    return date_str


def validate_tax_math(data):
    taxable  = data.get("taxable_amount", 0) or 0
    computed = round(taxable + (data.get("cgst",0) or 0)
                             + (data.get("sgst",0) or 0)
                             + (data.get("igst",0) or 0), 2)
    actual   = round(data.get("total_amount", 0) or 0, 2)
    return abs(computed - actual) <= 2.0, computed, actual


def check_duplicate(data, excel_file="gst_data.xlsx"):
    """
    v3 FIX: Uses display column names ('Invoice No.', 'Vendor Name')
    which is what Excel actually stores — NOT internal key names.
    Old code used 'invoice_number' which always returned None.
    """
    if not os.path.exists(excel_file): return False
    try:
        # header=2: skip title (row1) + subtitle (row2), use row3 as column headers
        df = pd.read_excel(excel_file, sheet_name="Invoices", header=2)
        df.columns = df.columns.str.strip()

        if "Vendor Name" not in df.columns or "Invoice No." not in df.columns:
            return False

        # Drop TOTAL row and blank rows
        df = df[
            df["Vendor Name"].notna() &
            (df["Vendor Name"].astype(str).str.strip() != "") &
            (df["S.No"].astype(str).str.strip() != "TOTAL")
        ]
        if df.empty: return False

        inv    = str(data.get("invoice_number", "")).strip().lower()
        vendor = str(data.get("vendor_name",    "")).strip().lower()

        for _, row in df.iterrows():
            # ✅ v3 FIX: use display column names
            if (str(row.get("Invoice No.", "")).strip().lower() == inv and
                str(row.get("Vendor Name", "")).strip().lower() == vendor):
                return True
        return False

    except Exception as e:
        print(f"  Duplicate check error: {e}"); return False


def validate_data(data, excel_file="gst_data.xlsx"):
    result = {"data": data, "warnings": [], "errors": [], "is_duplicate": False}
    if not data:
        result["errors"].append("No data extracted."); return result

    # [1] GSTIN
    valid_gstin = validate_gstin(data.get("gstin"))
    if data.get("gstin") and not valid_gstin:
        result["warnings"].append(f"Invalid GSTIN: '{data['gstin']}' — saved as null.")
    data["gstin"] = valid_gstin

    # [2] Date
    data["invoice_date"] = validate_date(data.get("invoice_date"))
    if not data["invoice_date"]:
        result["warnings"].append("Invoice date could not be parsed.")

    # [3] Tax math (warning only)
    ok, computed, actual = validate_tax_math(data)
    if not ok:
        result["warnings"].append(
            f"Tax math mismatch: ₹{computed:,.2f} ≠ ₹{actual:,.2f} "
            f"(diff ₹{abs(computed-actual):.2f}) — verify manually."
        )

    # [4] Required fields (blocks saving)
    for f in ["vendor_name", "invoice_number", "taxable_amount", "total_amount"]:
        if not data.get(f):
            result["errors"].append(f"Missing required field: '{f}'")

    # [5] Duplicate (blocks saving)
    if check_duplicate(data, excel_file):
        result["is_duplicate"] = True
        result["errors"].append(
            f"DUPLICATE: Invoice '{data.get('invoice_number')}' from "
            f"'{data.get('vendor_name')}' already exists. Skipped."
        )

    result["data"] = data
    return result


# ── Verify duplicate detection works ──
test_dup = validate_data({"vendor_name": "Test", "invoice_number": "INV-001",
                         "taxable_amount": 1000, "total_amount": 1180,
                          "cgst": 90, "sgst": 90, "igst": 0}, excel_file=EXCEL_FILE)
print(test_dup)

{'data': {'vendor_name': 'Test', 'invoice_number': 'INV-001', 'taxable_amount': 1000, 'total_amount': 1180, 'cgst': 90, 'sgst': 90, 'igst': 0, 'gstin': None, 'invoice_date': None}, 'warnings': ['Invoice date could not be parsed.'], 'errors': [], 'is_duplicate': False}


## 6️.Excel Output

In [14]:
import pandas as pd, os
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from datetime import datetime

HEADER_FILL  = PatternFill("solid", start_color="1F4E79")
TOTAL_FILL   = PatternFill("solid", start_color="D6E4F0")
ALT_FILL     = PatternFill("solid", start_color="F2F7FB")
WHITE_FILL   = PatternFill("solid", start_color="FFFFFF")
HEADER_FONT  = Font(bold=True, color="FFFFFF", name="Arial", size=10)
TOTAL_FONT   = Font(bold=True, name="Arial", size=10)
DATA_FONT    = Font(name="Arial", size=9)
THIN         = Side(style="thin", color="BFBFBF")
BORDER       = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
CURRENCY_FMT = '#,##0.00'
CENTER       = Alignment(horizontal="center", vertical="center")
LEFT         = Alignment(horizontal="left",   vertical="center")
RIGHT        = Alignment(horizontal="right",  vertical="center")

HEADERS       = ["S.No","Vendor Name","GSTIN","Invoice No.","Invoice Date",
                 "Taxable (₹)","CGST (₹)","SGST (₹)","IGST (₹)","Total (₹)","Processed On"]
WIDTHS        = [6, 28, 20, 18, 14, 16, 12, 12, 12, 16, 20]
CURRENCY_COLS = {6, 7, 8, 9, 10}


def _sc(cell, font=None, fill=None, alignment=None, border=None, num_format=None):
    if font:       cell.font          = font
    if fill:       cell.fill          = fill
    if alignment:  cell.alignment     = alignment
    if border:     cell.border        = border
    if num_format: cell.number_format = num_format


def _write_invoices_sheet(ws, df):
    ws.merge_cells("A1:K1")
    _sc(ws["A1"], Font(bold=True, color="1F4E79", name="Arial", size=14),
        PatternFill("solid", start_color="D6E4F0"), CENTER, BORDER)
    ws["A1"].value = "GST Invoice Register"
    ws.row_dimensions[1].height = 28

    ws.merge_cells("A2:K2")
    ws["A2"].value = f"Generated: {datetime.now().strftime('%d %b %Y, %H:%M')}"
    _sc(ws["A2"], Font(italic=True, color="595959", name="Arial", size=9), alignment=CENTER)
    ws.row_dimensions[2].height = 14

    for col, (h, w) in enumerate(zip(HEADERS, WIDTHS), start=1):
        cell = ws.cell(row=3, column=col, value=h)
        _sc(cell, HEADER_FONT, HEADER_FILL, CENTER, BORDER)
        ws.column_dimensions[get_column_letter(col)].width = w
    ws.row_dimensions[3].height = 20

    data_start = 4
    for i, (_, row) in enumerate(df.iterrows()):
        r    = data_start + i
        fill = ALT_FILL if i % 2 == 1 else WHITE_FILL
        vals = [i+1, row.get("vendor_name"), row.get("gstin"),
                row.get("invoice_number"),   row.get("invoice_date"),
                row.get("taxable_amount"),   row.get("cgst"),
                row.get("sgst"),             row.get("igst"),
                row.get("total_amount"),     row.get("processed_on")]
        for col, val in enumerate(vals, start=1):
            cell = ws.cell(row=r, column=col, value=val)
            align = CENTER if col == 1 else (RIGHT if col in CURRENCY_COLS else LEFT)
            _sc(cell, DATA_FONT, fill, align, BORDER,
                CURRENCY_FMT if col in CURRENCY_COLS else None)
        ws.row_dimensions[r].height = 17

    data_end   = data_start + len(df) - 1
    totals_row = data_end + 1
    col_map    = {6:"F", 7:"G", 8:"H", 9:"I", 10:"J"}
    for col in range(1, 12):
        cell = ws.cell(row=totals_row, column=col)
        _sc(cell, TOTAL_FONT, TOTAL_FILL, border=BORDER)
        if col == 1:
            cell.value = "TOTAL"; cell.alignment = CENTER
        elif col in col_map:
            L = col_map[col]
            cell.value         = f"=SUM({L}{data_start}:{L}{data_end})"
            cell.alignment     = RIGHT
            cell.number_format = CURRENCY_FMT
        else:
            cell.alignment = LEFT
    ws.row_dimensions[totals_row].height = 20
    ws.freeze_panes = "A4"


def _write_summary_sheet(ws, df):
    ws.column_dimensions["A"].width = 30
    ws.column_dimensions["B"].width = 22
    ws.sheet_view.showGridLines = False
    ws.merge_cells("A1:B1")
    ws["A1"].value = "GST Summary Report"
    _sc(ws["A1"], Font(bold=True, color="1F4E79", name="Arial", size=13),
        PatternFill("solid", start_color="D6E4F0"), CENTER, BORDER)
    ws.row_dimensions[1].height = 26

    rows = [
        ("Total Invoices Processed", len(df)),
        ("Total Taxable Amount",     df["taxable_amount"].sum() if "taxable_amount" in df else 0),
        ("Total CGST",               df["cgst"].sum()           if "cgst"           in df else 0),
        ("Total SGST",               df["sgst"].sum()           if "sgst"           in df else 0),
        ("Total IGST",               df["igst"].sum()           if "igst"           in df else 0),
        ("", ""),
        ("FINAL AMOUNT PAYABLE",     df["total_amount"].sum()   if "total_amount"   in df else 0),
    ]
    for r_idx, (label, value) in enumerate(rows, start=2):
        a = ws.cell(row=r_idx, column=1, value=label)
        b = ws.cell(row=r_idx, column=2, value=value)
        is_final = label == "FINAL AMOUNT PAYABLE"
        is_blank  = label == ""
        fill = (PatternFill("solid", start_color="1F4E79") if is_final else
                PatternFill("solid", start_color="F2F7FB") if not is_blank else WHITE_FILL)
        fc = "FFFFFF" if is_final else "000000"
        for cell in [a, b]:
            _sc(cell, Font(bold=is_final, name="Arial",
                           size=11 if is_final else 10, color=fc),
                fill, LEFT, BORDER if not is_blank else None)
        if isinstance(value, (int, float)) and not is_blank:
            b.number_format = CURRENCY_FMT; b.alignment = RIGHT
        ws.row_dimensions[r_idx].height = 22 if is_final else 18


def save_to_excel(record, file_name="gst_data.xlsx"):
    """Append one record and rewrite the full formatted workbook."""
    rec = {k: record.get(k) for k in
           ["vendor_name","gstin","invoice_number","invoice_date",
            "taxable_amount","cgst","sgst","igst","total_amount"]}
    rec["processed_on"] = datetime.now().strftime("%d-%m-%Y %H:%M")

    if os.path.exists(file_name):
        df = pd.read_excel(file_name, sheet_name="Invoices", header=2)
        df.columns = df.columns.str.strip()
        df = df[
            df["Vendor Name"].notna() &
            (df["Vendor Name"].astype(str).str.strip() != "") &
            (df["S.No"].astype(str).str.strip() != "TOTAL")
        ]
        df = df.rename(columns={
            "Vendor Name":  "vendor_name",  "GSTIN":        "gstin",
            "Invoice No.": "invoice_number", "Invoice Date": "invoice_date",
            "Taxable (₹)": "taxable_amount", "CGST (₹)":     "cgst",
            "SGST (₹)":    "sgst",           "IGST (₹)":     "igst",
            "Total (₹)":   "total_amount",   "Processed On": "processed_on",
        })
        df = df.drop(columns=["S.No"], errors="ignore")
    else:
        df = pd.DataFrame()

    df = pd.concat([df, pd.DataFrame([rec])], ignore_index=True)

    wb = Workbook()
    ws_inv = wb.active; ws_inv.title = "Invoices"
    _write_invoices_sheet(ws_inv, df)
    _write_summary_sheet(wb.create_sheet("Summary"), df)
    wb.save(file_name)
    print(f"  Saved → {file_name} ({len(df)} invoice(s))")


def add_summary(file_name="gst_data.xlsx"):
    df = pd.read_excel(file_name, sheet_name="Invoices", header=2)
    df.columns = df.columns.str.strip()
    df = df[
        df["Vendor Name"].notna() &
        (df["Vendor Name"].astype(str).str.strip() != "") &
        (df["S.No"].astype(str).str.strip() != "TOTAL")
    ]
    print("\n" + "═"*48)
    print("       GST INVOICE SUMMARY")
    print("═"*48)
    print(f"  Total Invoices    : {len(df)}")
    print(f"  Total Taxable     : ₹{df['Taxable (₹)'].sum():>14,.2f}")
    print(f"  Total CGST        : ₹{df['CGST (₹)'].sum():>14,.2f}")
    print(f"  Total SGST        : ₹{df['SGST (₹)'].sum():>14,.2f}")
    print(f"  Total IGST        : ₹{df['IGST (₹)'].sum():>14,.2f}")
    print("─"*48)
    print(f"  FINAL PAYABLE     : ₹{df['Total (₹)'].sum():>14,.2f}")
    print("═"*48)

## 7️.Single Invoice Pipeline

In [16]:
file_path = "invoice_1_electronics.pdf"  # ← Change to your invoice

print(f"Processing: {file_path}")
print("─" * 50)

# Step 1: Extract
text = extract_text(file_path)
print(f"[1] Extracted {len(text)} characters")

# Step 2: LLM parse
data = extract_invoice_data(text)
if data:
    print("[2] LLM output:")
    for k, v in data.items():
        print(f"      {k:20s}: {v}")

# Step 3: Validate
result = validate_data(data, excel_file=EXCEL_FILE)
for w in result["warnings"]: print(f"  ⚠  {w}")
for e in result["errors"]:   print(f"  ✘  {e}")

# Step 4: Save
if not result["errors"] and not result["is_duplicate"]:
    save_to_excel(result["data"], file_name=EXCEL_FILE)
    add_summary(EXCEL_FILE)
elif result["is_duplicate"]:
    print("\n[SKIP] Duplicate invoice — not saved.")
else:
    print("\n[SKIP] Validation failed — not saved.")

Processing: invoice_1_electronics.pdf
──────────────────────────────────────────────────
[1] Extracted 1455 characters
[2] LLM output:
      vendor_name         : Sri Murugan Electronics Pvt Ltd
      gstin               : 33AABCS1429B1Z8
      invoice_number      : SME/2025-26/1042
      invoice_date        : 05-01-2026
      taxable_amount      : 52399.0
      cgst                : 4715.91
      sgst                : 4715.91
      igst                : 0.0
      total_amount        : 61831.0
  ✘  DUPLICATE: Invoice 'SME/2025-26/1042' from 'Sri Murugan Electronics Pvt Ltd' already exists. Skipped.

[SKIP] Duplicate invoice — not saved.
